# 전국폐기물처리업소표준데이터 EDA

이 노트북은 전국 폐기물처리업소 데이터에서 **소각 가능 민간시설 후보**를 추려내기 위한 탐색 과정을 담습니다.

분석 원칙:
- 모든 본격 분석 전에 `head()`와 컬럼 목록을 먼저 확인합니다.
- 셀마다 왜 이 작업을 하는지 마크다운과 주석으로 설명합니다.
- 최종 목표는 대시보드 추천 로직에 사용할 민간 소각장 후보 테이블을 만드는 것입니다.

> **결론 미리보기**: 이 데이터셋은 `전문처리분야명` 컬럼의 결측률이 87.7%에 달해 소각장 필터링이 사실상 불가능했습니다. 이 한계를 확인한 후 notebook 04에서 경기도/인천/충남 개별 데이터를 합치는 접근으로 전환했습니다.

In [10]:
# 기본 라이브러리를 불러옵니다.
from pathlib import Path
import pandas as pd

DATA_DIR = Path('../../data/raw/소각장/폐기업체_데이터')
csv_path = DATA_DIR / '전국폐기물처리업소표준데이터.csv'

# 공공데이터 CSV는 종종 cp949/euc-kr 인코딩을 사용하므로 우선 cp949로 읽습니다.
df = pd.read_csv(csv_path, encoding='cp949')
print(f'파일명: {csv_path.name}')
print(f'행 x 열: {df.shape}')


파일명: 전국폐기물처리업소표준데이터.csv
행 x 열: (9112, 16)


## 1. 데이터 첫 확인

가장 먼저 데이터의 전반적인 형태를 확인합니다. 이 단계에서는 다음을 봅니다.
- 행/열 개수
- 컬럼명
- 자료형
- 상위 5개 행

In [11]:
# 모든 분석 전에 반드시 head()와 컬럼을 먼저 확인합니다.
print('shape:', df.shape)
print('\ncolumns:')
print(df.columns.tolist())
print('\ndtypes:')
print(df.dtypes)
display(df.head())

shape: (9112, 16)

columns:
['시설명', '소재지도로명주소', '소재지지번주소', '위도', '경도', '업종명', '전문처리분야명', '처리폐기물정보', '영업구역', '시설·장비명', '허가일자', '전화번호', '관리기관명', '데이터기준일자', '제공기관코드', '제공기관명']

dtypes:
시설명          object
소재지도로명주소     object
소재지지번주소      object
위도          float64
경도          float64
업종명          object
전문처리분야명      object
처리폐기물정보      object
영업구역         object
시설·장비명       object
허가일자         object
전화번호         object
관리기관명        object
데이터기준일자      object
제공기관코드        int64
제공기관명        object
dtype: object


,시설명,소재지도로명주소,소재지지번주소,위도,경도,업종명,전문처리분야명,처리폐기물정보,영업구역,시설·장비명,허가일자,전화번호,관리기관명,데이터기준일자,제공기관코드,제공기관명
0,동진산업,경상북도 청송군 현동면 청송로 2716-22,NaN,36.289193,129.024918,폐기물중간재활용업,파쇄,51-20-21,경상북도 청송군청,파쇄·분쇄시설,2013-07-17,054-872-8551,경상북도 청송군청,2025-12-24,5160000,경상북도 청송군
1,한라환경산업㈜,제주특별자치도 서귀포시 인정오름로 86번길 63,NaN,33.292406,126.576083,폐기물중간처분업,NaN,51-22-01+52-23-00+51-24-00+51-25-00+51-26-00+4...,NaN,NaN,2020-11-13,NaN,제주특별자치도 서귀포시청,2025-12-10,6520000,제주특별자치도 서귀포시
2,이도에코표선 주식회사,제주특별자치도 서귀포시 표선면 세성로 650,NaN,33.358520,126.798656,폐기물중간처분업,NaN,51-22-01+52-23-00+51-24-00+51-25-00+51-26-00+5...,NaN,NaN,2013-03-06,NaN,제주특별자치도 서귀포시청,2025-12-10,6520000,제주특별자치도 서귀포시
3,성진산업㈜,제주특별자치도 서귀포시 남원읍 위미항구로 502,NaN,33.317656,126.645419,폐기물중간처분업,NaN,51-22-01+52-23-00+51-24-00+51-25-00+51-26-00+4...,NaN,NaN,2012-06-12,NaN,제주특별자치도 서귀포시청,2025-12-10,6520000,제주특별자치도 서귀포시
4,일승산업㈜,제주특별자치도 서귀포시 성산읍 중산간동로 4364-64,NaN,33.391402,126.823953,폐기물중간처분업,NaN,51-22-01+52-23-00+51-24-00+51-25-00+51-26-00+4...,NaN,NaN,2011-01-18,NaN,제주특별자치도 서귀포시청,2025-12-10,6520000,제주특별자치도 서귀포시


## 2. 전체 행 수와 업종명 분포 확인

소각장만 따로 고르기 전에 전체 데이터에 어떤 업종이 섞여 있는지 확인합니다.
특히 `업종명` 컬럼의 unique 값을 먼저 보는 것이 중요합니다.

In [12]:
print('전체 행 수:', len(df))
print('\n업종명=== unique 값:')
for value in sorted(df['업종명'].dropna().astype(str).unique()):
    print(value)

전체 행 수: 9112

업종명=== unique 값:
폐기물수집·운반업
폐기물수집및운반업
폐기물종합재활용업
폐기물종합처분업
폐기물중간재활용업
폐기물중간재활용업+폐기물종합재활용업
폐기물중간처분업
폐기물최종재활용업
폐기물최종처분업


## 3. 소각 가능 업체 필터링

문제 정의에서 정한 조건에 따라 소각 가능 업체만 남깁니다.

필터 조건:
- 업종명: `폐기물중간처분업` 또는 `폐기물최종처분업`
- 처리폐기물정보: `52-23-00`, `52-24-00`, `52-25-00`, `52-26-00` 중 하나 이상 포함

In [16]:
import pandas as pd


# 1차: 업종명 필터
처분업종 = df['업종명'].isin([
    '폐기물중간처분업',
    '폐기물최종처분업',
    '폐기물종합처분업'
])

# 2차: 전문처리분야명 필터 (공백 포함 처리)
소각분야 = df['전문처리분야명'].fillna('').str.contains('소각|열분해')

# 조합: 업종이 처분업 AND (전문분야가 소각 OR 전문분야 공백)
소각업체 = df[처분업종 & (소각분야 | df['전문처리분야명'].fillna('') == '')]

# 전문분야 공백 포함하면 너무 많이 나올 수 있으니
# 우선 둘 다 충족하는 것만 먼저 확인
소각업체_확실 = df[처분업종 & 소각분야]

print(f"처분업종 전체: {처분업종.sum()}개")
print(f"소각 확실: {len(소각업체_확실)}개")
print(소각업체_확실[['시설명','소재지도로명주소','전문처리분야명','영업구역']].head(10))

처분업종 전체: 222개
소각 확실: 8개
                시설명                     소재지도로명주소    전문처리분야명 영업구역
2963  에이치엘비에너지 주식회사   부산광역시 사하구 감천항로291번길80(구평동)       일반소각   전국
2964        ㈜에너지네트웍       부산광역시 사하구 강변대로 20(신평동)       일반소각   전국
4031        한국환경개발㈜  경기도 안산시 단원구 첨단로207번길 5(성곡동)       일반소각  NaN
4032          대일개발㈜       경기도 안산시 단원구 지원로 7(성곡동)  일반소각+고온소각  NaN
4033          부경산업㈜  경기도 안산시 단원구 신원로91번길 16(성곡동)  일반소각+고온소각  NaN
4034           비노텍㈜     경기도 안산시 단원구 해안로 308(원시동)       일반소각  NaN
4035          성림유화㈜     경기도 안산시 단원구 첨단로 215(성곡동)  일반소각+고온소각  NaN
4731      (주)이엠케이승경   전북특별자치도 익산시 춘포면 궁성로 188-14       일반소각   전국


In [14]:
cond_52 = df['처리폐기물정보'].fillna("").str.contains(r'\b52-', regex=True)

print("52 포함 행 개수:", cond_52.sum())

52 포함 행 개수: 26


- 2023년 기준 소각시설이 245개인데 전국폐기물처리업소표준데이터로 필터링이 안됨(결측값이 너무 많음 …)
    - `전문처리분야명` 컬럼이 소각 여부를 가장 직접적으로 알 수 있는 컬럼인데, 데이터 명세서에 **"선택"** 항목으로 지정되어 있음. 처분업체 222개 중 전문처리분야명을 입력한 곳이 8개뿐(소각인지 아닌지 구분할 수 X)
    - `폐기물중간처분업`, `폐기물최종처분업`, `폐기물종합처분업` 3개가 소각 후보 업종이지만, 이 업종 안에는 소각 외에도 **매립, 압축, 파쇄, 고형화, 중화, 생물학적 처분** 등이 전부 포함
    - 처리폐기물코드 중 52(소각)이 포함된 행은 26개 뿐 ..

-> -> 서울 인근(인천/경기/충청)의 소각 데이터만 따로 수집 → [04_인천_경기_충남_폐기업체_통합](./04_인천_경기_충남_폐기업체_통합.ipynb)
